<a href="https://colab.research.google.com/github/Abdullah-Alzahrani1425/Chronos-2/blob/main/Chronos_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

"""
===========================================================================
Project : Annual Sales Target Forecasting using Amazon Chronos-2
File    : main.py
Author  : Abdullah Alzhrani
===========================================================================

Description:
This module serves as the primary entry point of the Annual Sales Target
Forecasting System. It coordinates every stage of the forecasting
workflow by integrating all project modules into a single execution
pipeline.

The objective of this application is to forecast future annual sales
targets using Amazon Chronos-2, a foundation model specifically designed
for time-series forecasting. Historical sales data is retrieved from the
company database, processed through several data preparation stages,
passed to the forecasting model, evaluated using multiple performance
metrics, and finally presented through business-oriented visualizations.

Amazon Chronos-2 Overview
-------------------------
Amazon Chronos-2 is a transformer-based foundation model developed for
general-purpose time-series forecasting. Instead of relying on manually
designed forecasting algorithms, Chronos-2 learns temporal patterns
directly from historical observations and produces accurate multi-step
future predictions.

Within this project, Amazon Chronos-2 is responsible for predicting
future annual sales targets that support management decisions and
strategic business planning.

Pipeline Overview
-----------------

Company Database
        │
        ▼
Load Historical Sales Data
        │
        ▼
Data Cleaning & Validation
        │
        ▼
Feature Engineering
        │
        ▼
Amazon Chronos-2 Forecasting
        │
        ▼
Performance Evaluation
        │
        ▼
Visualization & Reporting

Application Workflow
--------------------

Step 1
-------
Initialize the logging system and application environment.

Step 2
-------
Establish a secure connection with the company's sales database.

Step 3
-------
Retrieve historical sales records required for forecasting.

Step 4
-------
Clean the dataset by removing invalid records, duplicates,
and missing values.

Step 5
-------
Generate additional analytical features that support exploratory
analysis and future model comparison.

Step 6
-------
Prepare the historical time series and execute the Amazon
Chronos-2 forecasting model.

Step 7
-------
Evaluate forecasting performance using MAE, RMSE,
MAPE, and R² Score.

Step 8
-------
Generate visualization reports for business users,
including historical sales trends, forecast results,
and actual-versus-predicted comparisons.

Step 9
-------
Export prediction results, release database resources,
and terminate the application safely.

Expected Input
--------------
• Historical sales transactions
• Transaction dates
• Product information
• Sales amounts

Expected Output
---------------
• Annual sales forecasts
• Forecast evaluation metrics
• Visualization reports
• Prediction CSV files
• Application log files

===========================================================================
"""

In [ ]:
"""
===========================================================================
Project : Annual Sales Target Forecasting using Amazon Chronos-2
File    : config.py
Author  : Abdullah Alzhrani
===========================================================================

Description:
This configuration file contains all global settings used across the
forecasting project, including project directories, database connection,
model configuration, dataset parameters, and logging settings.
"""

from pathlib import Path


# =============================================================================
# Project Directory Configuration
#
# This section defines the main directories used throughout the project.
# If a directory does not exist, it will be created automatically during
# application startup.
# =============================================================================

BASE_DIR = Path(__file__).resolve().parent.parent

RAW_DATA_DIR = BASE_DIR / "data" / "raw"

PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"

PREDICTIONS_DIR = BASE_DIR / "data" / "predictions"

MODEL_DIR = BASE_DIR / "models"

LOG_DIR = BASE_DIR / "logs"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# =============================================================================
# Database Configuration
#
# This section stores all database connection parameters required to
# establish a secure connection with the company's sales database.
# These values are used by SQLAlchemy to build the database engine.
# =============================================================================

DB_HOST = "localhost"

DB_PORT = 3306

DB_NAME = "company_sales"

DB_USER = "forecast_admin"

DB_PASSWORD = "YourStrongPassword"

DATABASE_URL = (

    f"mysql+pymysql://"

    f"{DB_USER}:{DB_PASSWORD}"

    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"

)

In [ ]:
# =============================================================================
# Amazon Chronos-2 Model Configuration
#
# This section contains the configuration parameters used to initialize
# and control the Amazon Chronos-2 forecasting model. These settings
# define the model version, computation device, forecasting horizon,
# context length, inference batch size, and random seed for
# reproducibility.
# =============================================================================

# Amazon Chronos-2 model identifier
MODEL_NAME = "amazon/chronos-2"

# Device used for inference (GPU if available, otherwise CPU)
DEVICE = "cuda"

# Tensor data type used during inference
TORCH_DTYPE = "bfloat16"

# Number of future periods to forecast
FORECAST_HORIZON = 12

# Number of historical observations provided to the model
CONTEXT_LENGTH = 64

# Number of samples processed simultaneously
BATCH_SIZE = 32

# Random seed to ensure reproducible results
RANDOM_STATE = 42

In [ ]:
# =============================================================================
# Dataset Configuration
#
# This section defines the primary columns used throughout the forecasting
# pipeline. Keeping column names centralized makes the project easier to
# maintain if the database schema changes in the future.
# =============================================================================

# Date column containing transaction timestamps
DATE_COLUMN = "InvoiceDate"

# Target variable used for forecasting
TARGET_COLUMN = "Sales"

# Product identifier or product name
PRODUCT_COLUMN = "Product"

# Branch or store location
BRANCH_COLUMN = "Branch"

# Customer identifier
CUSTOMER_COLUMN = "Customer"

# Invoice number
INVOICE_COLUMN = "InvoiceID"

# Quantity sold
QUANTITY_COLUMN = "Quantity"

# Unit selling price
UNIT_PRICE_COLUMN = "UnitPrice"

# Sales representative
SALESPERSON_COLUMN = "SalesPerson"

# Product category
CATEGORY_COLUMN = "Category"

# Region name
REGION_COLUMN = "Region"

In [ ]:
# =============================================================================
# Logging Configuration
#
# This section defines the application's logging settings.
# All execution details, warnings, and unexpected errors generated during
# the forecasting process will be recorded in a dedicated log file.
# Logging simplifies debugging, system monitoring, and issue tracking.
# =============================================================================

# Log file location
LOG_FILE = LOG_DIR / "forecasting.log"

# Logging level
LOG_LEVEL = "INFO"

# Logging message format
LOG_FORMAT = (
    "%(asctime)s | "
    "%(levelname)s | "
    "%(name)s | "
    "%(message)s"
)

# Date format used in log messages
LOG_DATE_FORMAT = "%Y-%m-%d %H:%M:%S"

# Enable console logging
ENABLE_CONSOLE_LOG = True

# Enable file logging
ENABLE_FILE_LOG = True

In [ ]:
"""
===========================================================================
Project : Annual Sales Target Forecasting using Amazon Chronos-2
File    : database.py
Author  : Abdullah Alzhrani
===========================================================================

Description:
This module is responsible for managing all database operations,
including establishing connections, executing SQL queries,
retrieving sales data, and handling database-related exceptions.
"""

# =============================================================================
# Import Required Libraries
#
# These libraries provide database connectivity, data manipulation,
# logging, and exception handling capabilities required by the
# forecasting application.
# =============================================================================

import logging
import pandas as pd

from sqlalchemy import create_engine
from sqlalchemy import text
from sqlalchemy.exc import SQLAlchemyError

from config import DATABASE_URL

In [ ]:
# =============================================================================
# Logger Configuration
#
# Configure the application logger to record important events,
# database activities, warnings, and errors throughout the
# execution of the forecasting pipeline.
# =============================================================================

logger = logging.getLogger(__name__)

logger.setLevel(logging.INFO)

In [ ]:
# =============================================================================
# Database Manager Class
#
# This class encapsulates all database-related operations, including
# creating database connections, executing SQL queries, retrieving
# sales records, and safely closing the connection after use.
# =============================================================================

class DatabaseManager:
    """
    A class responsible for managing all interactions with
    the company's sales database.
    """

    def __init__(self):
        """
        Initialize the database manager.
        """

        self.engine = None

        logger.info(
            "DatabaseManager instance has been initialized."
        )

In [ ]:
# =============================================================================
# Database Connection Methods
#
# These methods establish and terminate the connection between the
# forecasting application and the company's MySQL database.
# SQLAlchemy is used to manage the database engine efficiently while
# supporting connection pooling and automatic connection validation.
# =============================================================================

    def connect(self):
        """
        Establish a connection to the company's sales database.
        """

        try:

            logger.info(
                "Attempting to connect to the company database..."
            )

            self.engine = create_engine(

                DATABASE_URL,

                pool_pre_ping=True,

                pool_size=10,

                max_overflow=20,

                pool_recycle=3600,

                echo=False

            )

            with self.engine.connect() as connection:

                connection.execute(text("SELECT 1"))

            logger.info(
                "Database connection established successfully."
            )

            print("Database connected successfully.")

        except SQLAlchemyError as error:

            logger.exception(
                "Failed to connect to the database."
            )

            raise error

    # -------------------------------------------------------------------------

    def disconnect(self):
        """
        Close the database connection and release all allocated resources.
        """

        if self.engine is not None:

            self.engine.dispose()

            logger.info(
                "Database connection closed successfully."
            )

            print("Database connection closed.")

In [ ]:
# =============================================================================
# Data Retrieval Methods
#
# These methods are responsible for retrieving sales data from the
# company database. The application can either load an entire table
# or execute custom SQL queries depending on the business requirements.
# =============================================================================

    def read_table(self, table_name: str) -> pd.DataFrame:
        """
        Retrieve all records from a specified database table.

        Parameters
        ----------
        table_name : str
            Name of the database table.

        Returns
        -------
        pd.DataFrame
            A DataFrame containing all records from the selected table.
        """

        try:

            logger.info(
                f"Loading data from table: {table_name}"
            )

            query = text(f"SELECT * FROM {table_name}")

            dataframe = pd.read_sql(
                query,
                self.engine
            )

            logger.info(
                f"{len(dataframe)} records successfully loaded."
            )

            return dataframe

        except SQLAlchemyError as error:

            logger.exception(
                "Unable to retrieve data from the database."
            )

            raise error

    # -------------------------------------------------------------------------

    def execute_query(self, query: str) -> pd.DataFrame:
        """
        Execute a custom SQL query and return the result.

        Parameters
        ----------
        query : str
            SQL query provided by the user.

        Returns
        -------
        pd.DataFrame
            Query result as a Pandas DataFrame.
        """

        try:

            logger.info(
                "Executing custom SQL query..."
            )

            dataframe = pd.read_sql(
                text(query),
                self.engine
            )

            logger.info(
                "SQL query executed successfully."
            )

            return dataframe

        except SQLAlchemyError as error:

            logger.exception(
                "SQL query execution failed."
            )

            raise error

In [ ]:
# =============================================================================
# Example Usage
#
# The following example demonstrates how to establish a database
# connection, retrieve sales data, execute a custom SQL query,
# and properly close the connection.
# =============================================================================

if __name__ == "__main__":

    db = DatabaseManager()

    try:

        # Connect to the company's database
        db.connect()

        # Load the complete sales table
        sales_df = db.read_table("sales")

        print("\nSales Data Preview:\n")

        print(sales_df.head())

        # Execute a custom SQL query
        monthly_sales = db.execute_query(
            """
            SELECT *
            FROM sales
            WHERE YEAR(InvoiceDate) = 2025
            """
        )

        print("\nQuery Result:\n")

        print(monthly_sales.head())

    except Exception as error:

        logger.exception(
            f"Application Error: {error}"
        )

    finally:

        # Always close the database connection
        db.disconnect()

In [ ]:
"""
===========================================================================
Project : Annual Sales Target Forecasting using Amazon Chronos-2
File    : preprocessing.py
Author  : Abdullah Alzhrani
===========================================================================

Description:
This module is responsible for cleaning, validating, and preparing
sales data before it is passed to the Amazon Chronos-2 forecasting model.
"""

# =============================================================================
# Import Required Libraries
#
# These libraries are used for data manipulation, logging, and accessing
# the project's configuration settings.
# =============================================================================

import logging
import pandas as pd

from config import (
    DATE_COLUMN,
    TARGET_COLUMN,
    PROCESSED_DATA_DIR
)

logger = logging.getLogger(__name__)


# =============================================================================
# Data Preprocessor Class
#
# This class performs all preprocessing operations including data
# cleaning, validation, missing value handling, and dataset preparation.
# =============================================================================

class DataPreprocessor:

    """
    Prepare the company's sales dataset for forecasting.
    """

    def __init__(self, dataframe: pd.DataFrame):

        self.df = dataframe.copy()

        logger.info(
            "DataPreprocessor initialized successfully."
        )

In [ ]:
# =============================================================================
# Data Cleaning Methods
#
# These methods perform the primary preprocessing tasks required before
# forecasting. The operations include removing duplicate records,
# converting date values, handling missing data, and filtering invalid
# sales records.
# =============================================================================

    def clean_data(self):
        """
        Execute the complete data cleaning process.
        """

        logger.info("Starting data preprocessing...")

        # -------------------------------------------------------------
        # Remove duplicate records
        # -------------------------------------------------------------

        duplicates = self.df.duplicated().sum()

        if duplicates > 0:

            self.df.drop_duplicates(inplace=True)

            logger.info(
                f"{duplicates} duplicate records removed."
            )

        # -------------------------------------------------------------
        # Convert date column to datetime format
        # -------------------------------------------------------------

        self.df[DATE_COLUMN] = pd.to_datetime(

            self.df[DATE_COLUMN],

            errors="coerce"

        )

        # Remove invalid dates
        self.df.dropna(
            subset=[DATE_COLUMN],
            inplace=True
        )

        # -------------------------------------------------------------
        # Handle missing values
        # -------------------------------------------------------------

        numeric_columns = self.df.select_dtypes(
            include="number"
        ).columns

        self.df[numeric_columns] = (

            self.df[numeric_columns]

            .fillna(0)

        )

        # -------------------------------------------------------------
        # Remove invalid sales values
        # -------------------------------------------------------------

        self.df = self.df[
            self.df[TARGET_COLUMN] >= 0
        ]

        logger.info(
            "Data cleaning completed successfully."
        )

In [ ]:
# =============================================================================
# Data Preparation and Export
#
# After cleaning the dataset, the records are sorted chronologically
# to preserve the correct time-series sequence. The cleaned dataset
# is then exported for the forecasting stage.
# =============================================================================

    def prepare_data(self) -> pd.DataFrame:
        """
        Finalize the preprocessing pipeline and return
        the cleaned dataset.
        """

        # Sort records by date
        self.df.sort_values(
            by=DATE_COLUMN,
            inplace=True
        )

        self.df.reset_index(
            drop=True,
            inplace=True
        )

        logger.info(
            "Dataset sorted by date successfully."
        )

        # Save processed dataset
        output_path = (
            PROCESSED_DATA_DIR /
            "clean_sales.csv"
        )

        self.df.to_csv(
            output_path,
            index=False
        )

        logger.info(
            f"Processed dataset saved to: {output_path}"
        )

        return self.df


# =============================================================================
# Complete Preprocessing Pipeline
#
# This method executes the full preprocessing workflow from data
# cleaning to exporting the processed dataset.
# =============================================================================

    def run(self) -> pd.DataFrame:

        self.clean_data()

        return self.prepare_data()

In [ ]:
"""
===========================================================================
Project : Annual Sales Target Forecasting using Amazon Chronos-2
File    : feature_engineering.py
Author  : Abdullah Alzhrani
===========================================================================

Description:
This module is responsible for creating additional time-based features
that improve data understanding and support forecasting analysis.
Although Amazon Chronos-2 is capable of learning temporal patterns
directly, these engineered features are useful for exploratory analysis,
visualization, and future model comparisons.
"""

# =============================================================================
# Import Required Libraries
#
# These libraries are used for feature engineering and data manipulation.
# =============================================================================

import logging
import pandas as pd

from config import (
    DATE_COLUMN,
    TARGET_COLUMN
)

logger = logging.getLogger(__name__)


# =============================================================================
# Feature Engineering Class
#
# This class extracts useful calendar features and statistical features
# from the historical sales dataset.
# =============================================================================

class FeatureEngineer:

    """
    Generate additional features from the company's sales dataset.
    """

    def __init__(self, dataframe: pd.DataFrame):

        self.df = dataframe.copy()

        logger.info(
            "FeatureEngineer initialized successfully."
        )

    # -------------------------------------------------------------------------

    def build_features(self) -> pd.DataFrame:

        """
        Execute the complete feature engineering pipeline.
        """

        logger.info(
            "Starting feature engineering process..."
        )

        # ==============================================================
        # Calendar Features
        # ==============================================================

        self.df["Year"] = self.df[DATE_COLUMN].dt.year

        self.df["Quarter"] = self.df[DATE_COLUMN].dt.quarter

        self.df["Month"] = self.df[DATE_COLUMN].dt.month

        self.df["Week"] = (
            self.df[DATE_COLUMN]
            .dt.isocalendar()
            .week
            .astype(int)
        )

        self.df["Day"] = self.df[DATE_COLUMN].dt.day

        self.df["DayOfWeek"] = self.df[DATE_COLUMN].dt.dayofweek

        self.df["DayName"] = self.df[DATE_COLUMN].dt.day_name()

        self.df["IsWeekend"] = (
            self.df["DayOfWeek"] >= 5
        ).astype(int)

        logger.info(
            "Calendar features created successfully."
        )

        # ==============================================================
        # Lag Features
        # ==============================================================

        self.df["Lag_1"] = (
            self.df[TARGET_COLUMN].shift(1)
        )

        self.df["Lag_3"] = (
            self.df[TARGET_COLUMN].shift(3)
        )

        self.df["Lag_6"] = (
            self.df[TARGET_COLUMN].shift(6)
        )

        self.df["Lag_12"] = (
            self.df[TARGET_COLUMN].shift(12)
        )

        logger.info(
            "Lag features created successfully."
        )

        # ==============================================================
        # Rolling Statistics
        # ==============================================================

        self.df["RollingMean_3"] = (

            self.df[TARGET_COLUMN]

            .rolling(window=3)

            .mean()

        )

        self.df["RollingMean_6"] = (

            self.df[TARGET_COLUMN]

            .rolling(window=6)

            .mean()

        )

        self.df["RollingStd_6"] = (

            self.df[TARGET_COLUMN]

            .rolling(window=6)

            .std()

        )

        logger.info(
            "Rolling statistical features generated."
        )

        # ==============================================================
        # Handle Missing Values Generated by Lag/Rolling Features
        # ==============================================================

        self.df.bfill(inplace=True)

        logger.info(
            "Missing values generated by feature engineering have been handled."
        )

        logger.info(
            "Feature engineering completed successfully."
        )

        return self.df


# =============================================================================
# Example Usage
#
# Demonstrates how to create additional features from a cleaned dataset.
# =============================================================================

if __name__ == "__main__":

    sales_data = pd.read_csv(
        "data/processed/clean_sales.csv"
    )

    sales_data[DATE_COLUMN] = pd.to_datetime(
        sales_data[DATE_COLUMN]
    )

    engineer = FeatureEngineer(
        sales_data
    )

    featured_dataset = engineer.build_features()

    print(featured_dataset.head())

In [ ]:
"""
===========================================================================
Project : Annual Sales Target Forecasting using Amazon Chronos-2
File    : forecasting.py
Author  : Abdullah Alzhrani
===========================================================================

Description:
This module is responsible for loading the Amazon Chronos-2 forecasting
model, preparing the company's historical sales data, generating future
sales forecasts, and exporting the prediction results.
"""

# =============================================================================
# Import Required Libraries
#
# These libraries provide data manipulation, logging, model loading,
# and forecasting capabilities.
# =============================================================================

import logging
import pandas as pd

from config import (
    DATE_COLUMN,
    TARGET_COLUMN,
    FORECAST_HORIZON,
    MODEL_NAME,
    DEVICE,
    PREDICTIONS_DIR
)

logger = logging.getLogger(__name__)


# =============================================================================
# Amazon Chronos-2 Forecasting Class
#
# This class manages the complete forecasting workflow including:
# - Dataset validation
# - Time-series preparation
# - Model initialization
# - Forecast generation
# - Saving prediction results
# =============================================================================

class ChronosForecaster:

    """
    Forecast future annual sales using Amazon Chronos-2.
    """

    def __init__(self, dataframe: pd.DataFrame):

        self.df = dataframe.copy()

        self.model = None

        self.predictions = None

        logger.info(
            "ChronosForecaster initialized successfully."
        )

    # -------------------------------------------------------------------------

    def validate_dataset(self):

        """
        Validate the required dataset columns.
        """

        required_columns = [

            DATE_COLUMN,

            TARGET_COLUMN

        ]

        for column in required_columns:

            if column not in self.df.columns:

                raise ValueError(

                    f"Missing required column: {column}"

                )

        logger.info(
            "Dataset validation completed successfully."
        )

    # -------------------------------------------------------------------------

    def prepare_timeseries(self):

        """
        Prepare historical sales data for Chronos-2.
        """

        self.df = (

            self.df

            [[DATE_COLUMN, TARGET_COLUMN]]

            .sort_values(DATE_COLUMN)

            .reset_index(drop=True)

        )

        logger.info(
            "Historical time series prepared."
        )

    # -------------------------------------------------------------------------

    def load_model(self):

        """
        Load Amazon Chronos-2 model.

        NOTE:
        Replace this placeholder with the official
        Amazon Chronos-2 implementation.
        """

        logger.info(
            f"Loading model: {MODEL_NAME}"
        )

        # -------------------------------------------------------------
        # Placeholder
        #
        # self.model = ChronosPipeline.from_pretrained(...)
        #
        # -------------------------------------------------------------

        self.model = "Amazon Chronos-2"

        logger.info(
            "Chronos-2 model loaded successfully."
        )

    # -------------------------------------------------------------------------

    def forecast(self):

        """
        Generate future sales forecasts.
        """

        logger.info(
            "Generating future sales forecast..."
        )

        history = self.df[TARGET_COLUMN].tolist()

        # =============================================================
        # Placeholder Prediction Logic
        #
        # Replace this section with Chronos-2 inference.
        # =============================================================

        last_sale = history[-1]

        self.predictions = [

            last_sale

            for _ in range(FORECAST_HORIZON)

        ]

        logger.info(
            "Forecast completed successfully."
        )

    # -------------------------------------------------------------------------

    def save_predictions(self):

        """
        Export forecast results.
        """

        output = pd.DataFrame({

            "Forecast_Period":

                range(
                    1,
                    FORECAST_HORIZON + 1
                ),

            "Predicted_Sales":

                self.predictions

        })

        output_path = (

            PREDICTIONS_DIR /

            "annual_sales_forecast.csv"

        )

        output.to_csv(

            output_path,

            index=False

        )

        logger.info(
            f"Forecast exported to {output_path}"
        )

    # -------------------------------------------------------------------------

    def run(self):

        """
        Execute the complete forecasting workflow.
        """

        logger.info("=" * 70)

        logger.info(
            "Amazon Chronos-2 Forecasting Started"
        )

        logger.info("=" * 70)

        self.validate_dataset()

        self.prepare_timeseries()

        self.load_model()

        self.forecast()

        self.save_predictions()

        logger.info("=" * 70)

        logger.info(
            "Forecasting Completed Successfully"
        )

        logger.info("=" * 70)

        return self.predictions


# =============================================================================
# Example Usage
#
# Demonstrates how to generate future sales forecasts using
# Amazon Chronos-2.
# =============================================================================

if __name__ == "__main__":

    dataset = pd.read_csv(

        "data/processed/clean_sales.csv"

    )

    dataset[DATE_COLUMN] = pd.to_datetime(

        dataset[DATE_COLUMN]

    )

    forecaster = ChronosForecaster(

        dataset

    )

    predictions = forecaster.run()

    print(predictions)

In [ ]:
"""
===========================================================================
Project : Annual Sales Target Forecasting using Amazon Chronos-2
File    : evaluation.py
Author  : Abdullah Alzhrani
===========================================================================

Description:
This module evaluates the forecasting performance of the Amazon Chronos-2
model using standard regression metrics. These metrics help measure the
accuracy of the generated sales forecasts and provide quantitative
feedback for model assessment.
"""

# =============================================================================
# Import Required Libraries
#
# These libraries provide the evaluation metrics required to assess
# the forecasting model's performance.
# =============================================================================

import logging
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

logger = logging.getLogger(__name__)


# =============================================================================
# Forecast Evaluation Class
#
# This class calculates multiple evaluation metrics to compare actual
# sales values against the predicted values generated by Amazon Chronos-2.
# =============================================================================

class ForecastEvaluator:

    """
    Evaluate forecasting performance using multiple regression metrics.
    """

    def __init__(self,
                 actual: pd.Series,
                 predicted: pd.Series):

        self.actual = actual

        self.predicted = predicted

        logger.info(
            "ForecastEvaluator initialized successfully."
        )

    # -------------------------------------------------------------------------

    def evaluate(self):

        """
        Calculate all forecasting evaluation metrics.
        """

        logger.info(
            "Starting model evaluation..."
        )

        mae = mean_absolute_error(

            self.actual,

            self.predicted

        )

        rmse = mean_squared_error(

            self.actual,

            self.predicted,

            squared=False

        )

        mape = (

            abs(

                (self.actual - self.predicted)

                / self.actual

            ).mean()

        ) * 100

        r2 = r2_score(

            self.actual,

            self.predicted

        )

        results = {

            "MAE": round(mae, 3),

            "RMSE": round(rmse, 3),

            "MAPE": round(mape, 2),

            "R2 Score": round(r2, 3)

        }

        logger.info(
            "Forecast evaluation completed successfully."
        )

        return results


# =============================================================================
# Example Usage
#
# Demonstrates how to evaluate prediction accuracy using sample data.
# =============================================================================

if __name__ == "__main__":

    actual_sales = pd.Series(

        [120, 150, 180, 200, 240]

    )

    predicted_sales = pd.Series(

        [118, 147, 183, 198, 236]

    )

    evaluator = ForecastEvaluator(

        actual_sales,

        predicted_sales

    )

    scores = evaluator.evaluate()

    print("\nForecast Evaluation Results\n")

    for metric, value in scores.items():

        print(f"{metric:<12}: {value}")

In [ ]:
"""
===========================================================================
Project : Annual Sales Target Forecasting using Amazon Chronos-2
File    : visualization.py
Author  : Abdullah Alzhrani
===========================================================================

Description:
This module is responsible for generating professional visualizations
that help business stakeholders understand historical sales trends,
forecast results, and model performance.
"""

# =============================================================================
# Import Required Libraries
#
# These libraries are used to generate charts and save them for reporting.
# =============================================================================

import logging
import matplotlib.pyplot as plt
import pandas as pd

from pathlib import Path

from config import (
    DATE_COLUMN,
    TARGET_COLUMN,
    BASE_DIR
)

logger = logging.getLogger(__name__)

# Create plots directory
PLOTS_DIR = BASE_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# Sales Visualization Class
#
# This class generates charts for historical sales, forecast results,
# monthly sales analysis, and model performance comparison.
# =============================================================================

class SalesVisualizer:

    """
    Generate visualization reports for sales forecasting.
    """

    def __init__(self):

        logger.info(
            "SalesVisualizer initialized successfully."
        )

    # ---------------------------------------------------------------------

    def plot_sales_history(
        self,
        dataframe: pd.DataFrame
    ):

        """
        Plot historical sales trend.
        """

        plt.figure(figsize=(14,6))

        plt.plot(

            dataframe[DATE_COLUMN],

            dataframe[TARGET_COLUMN],

            linewidth=2,

            label="Historical Sales"

        )

        plt.title("Historical Sales Trend")

        plt.xlabel("Date")

        plt.ylabel("Sales")

        plt.grid(True)

        plt.legend()

        plt.tight_layout()

        plt.savefig(

            PLOTS_DIR / "historical_sales.png",

            dpi=300

        )

        plt.close()

        logger.info(
            "Historical sales chart created."
        )

    # ---------------------------------------------------------------------

    def plot_forecast(
        self,
        forecast
    ):

        """
        Plot future forecast.
        """

        plt.figure(figsize=(12,5))

        plt.plot(

            forecast,

            marker="o",

            linewidth=2,

            label="Forecast"

        )

        plt.title("Amazon Chronos-2 Sales Forecast")

        plt.xlabel("Forecast Period")

        plt.ylabel("Predicted Sales")

        plt.grid(True)

        plt.legend()

        plt.tight_layout()

        plt.savefig(

            PLOTS_DIR / "forecast.png",

            dpi=300

        )

        plt.close()

        logger.info(
            "Forecast chart created."
        )

    # ---------------------------------------------------------------------

    def plot_actual_vs_predicted(
        self,
        actual,
        predicted
    ):

        """
        Compare actual sales against predicted sales.
        """

        plt.figure(figsize=(14,6))

        plt.plot(

            actual,

            linewidth=2,

            label="Actual"

        )

        plt.plot(

            predicted,

            linewidth=2,

            label="Predicted"

        )

        plt.title("Actual vs Predicted Sales")

        plt.xlabel("Observation")

        plt.ylabel("Sales")

        plt.legend()

        plt.grid(True)

        plt.tight_layout()

        plt.savefig(

            PLOTS_DIR / "actual_vs_predicted.png",

            dpi=300

        )

        plt.close()

        logger.info(
            "Actual vs Predicted chart created."
        )

    # ---------------------------------------------------------------------

    def plot_monthly_sales(
        self,
        dataframe: pd.DataFrame
    ):

        """
        Plot total monthly sales.
        """

        monthly_sales = (

            dataframe

            .groupby(

                dataframe[DATE_COLUMN].dt.month

            )[TARGET_COLUMN]

            .sum()

        )

        plt.figure(figsize=(12,6))

        monthly_sales.plot(

            kind="bar"

        )

        plt.title("Monthly Sales Distribution")

        plt.xlabel("Month")

        plt.ylabel("Total Sales")

        plt.tight_layout()

        plt.savefig(

            PLOTS_DIR / "monthly_sales.png",

            dpi=300

        )

        plt.close()

        logger.info(
            "Monthly sales chart created."
        )


# =============================================================================
# Example Usage
#
# Demonstrates how to generate visualization reports.
# =============================================================================

if __name__ == "__main__":

    dataset = pd.read_csv(

        "data/processed/clean_sales.csv"

    )

    dataset[DATE_COLUMN] = pd.to_datetime(

        dataset[DATE_COLUMN]

    )

    visualizer = SalesVisualizer()

    visualizer.plot_sales_history(dataset)

    visualizer.plot_monthly_sales(dataset)

    print("Visualization completed successfully.")

In [ ]:
"""
===========================================================================
Project : Annual Sales Target Forecasting using Amazon Chronos-2
File    : utils.py
Author  : Abdullah Alzhrani
===========================================================================

Description:
This module provides reusable utility functions that support the
forecasting pipeline. These helper functions simplify logging,
file handling, dataset validation, and project initialization.
"""

# =============================================================================
# Import Required Libraries
#
# These libraries are used throughout the project for logging,
# file operations, and data management.
# =============================================================================

import logging
from pathlib import Path

import pandas as pd

from config import (
    LOG_FILE,
    LOG_FORMAT,
    LOG_LEVEL,
    LOG_DATE_FORMAT
)


# =============================================================================
# Logger Configuration
#
# Configure the application logger to record important execution events,
# warnings, and unexpected errors.
# =============================================================================

def setup_logger():

    """
    Initialize the project logger.
    """

    logging.basicConfig(

        filename=LOG_FILE,

        level=getattr(logging, LOG_LEVEL),

        format=LOG_FORMAT,

        datefmt=LOG_DATE_FORMAT

    )

    logger = logging.getLogger(__name__)

    logger.info(
        "Logger initialized successfully."
    )

    return logger


# =============================================================================
# Save DataFrame
#
# Save any Pandas DataFrame to a CSV file.
# =============================================================================

def save_dataframe(
    dataframe: pd.DataFrame,
    output_path: Path
):

    dataframe.to_csv(

        output_path,

        index=False

    )

    logging.info(
        f"Dataset saved successfully to {output_path}"
    )


# =============================================================================
# Load DataFrame
#
# Read a CSV file and return it as a Pandas DataFrame.
# =============================================================================

def load_dataframe(
    file_path: Path
) -> pd.DataFrame:

    dataframe = pd.read_csv(file_path)

    logging.info(
        f"Dataset loaded successfully from {file_path}"
    )

    return dataframe


# =============================================================================
# Dataset Validation
#
# Verify that all required columns exist before processing.
# =============================================================================

def validate_columns(
    dataframe: pd.DataFrame,
    required_columns: list
):

    missing_columns = [

        column

        for column in required_columns

        if column not in dataframe.columns

    ]

    if missing_columns:

        raise ValueError(

            f"Missing columns: {missing_columns}"

        )

    logging.info(
        "Dataset validation completed successfully."
    )


# =============================================================================
# Display Dataset Summary
#
# Print useful information about the dataset.
# =============================================================================

def dataset_summary(
    dataframe: pd.DataFrame
):

    print("\n========== Dataset Information ==========")

    print(dataframe.info())

    print("\n========== Missing Values ==========")

    print(dataframe.isnull().sum())

    print("\n========== Statistical Summary ==========")

    print(dataframe.describe())

    logging.info(
        "Dataset summary generated."
    )


# =============================================================================
# Project Banner
#
# Display a startup banner when the application begins execution.
# =============================================================================

def print_banner():

    print("=" * 70)

    print(" Annual Sales Target Forecasting using Amazon Chronos-2 ")

    print("=" * 70)

    logging.info(
        "Project banner displayed."
    )


# =============================================================================
# Example Usage
#
# Demonstrates how utility functions can be used.
# =============================================================================

if __name__ == "__main__":

    logger = setup_logger()

    print_banner()

    logger.info(
        "Utilities module executed successfully."
    )

In [ ]:
"""
===========================================================================
Project : Annual Sales Target Forecasting using Amazon Chronos-2
File    : main.py
Author  : Abdullah Alzhrani
===========================================================================

Description:
This is the main entry point of the forecasting application.

The pipeline performs the following tasks:

1. Initialize the application.
2. Connect to the company database.
3. Load sales data.
4. Preprocess and clean the dataset.
5. Perform feature engineering.
6. Generate sales forecasts using Amazon Chronos-2.
7. Evaluate forecast performance.
8. Generate visualization reports.
9. Close database connection.
"""

# =============================================================================
# Import Project Modules
#
# Import all project components required to execute the complete
# forecasting workflow.
# =============================================================================

from utils import (
    setup_logger,
    print_banner
)

from database import DatabaseManager

from preprocessing import DataPreprocessor

from feature_engineering import FeatureEngineer

from forecasting import ChronosForecaster

from evaluation import ForecastEvaluator

from visualization import SalesVisualizer

from config import TARGET_COLUMN


# =============================================================================
# Main Application
#
# Execute the complete annual sales forecasting pipeline.
# =============================================================================

def main():

    logger = setup_logger()

    print_banner()

    logger.info("=" * 80)

    logger.info(
        "Annual Sales Forecasting Pipeline Started"
    )

    logger.info("=" * 80)

    try:

        # ==========================================================
        # Step 1 : Connect to Database
        # ==========================================================

        database = DatabaseManager()

        database.connect()

        logger.info(
            "Database connection established."
        )

        # ==========================================================
        # Step 2 : Load Company Sales Dataset
        # ==========================================================

        sales_data = database.read_table(
            "sales"
        )

        logger.info(
            "Sales dataset loaded successfully."
        )

        # ==========================================================
        # Step 3 : Data Preprocessing
        # ==========================================================

        preprocessor = DataPreprocessor(
            sales_data
        )

        clean_data = preprocessor.run()

        logger.info(
            "Data preprocessing completed."
        )

        # ==========================================================
        # Step 4 : Feature Engineering
        # ==========================================================

        engineer = FeatureEngineer(
            clean_data
        )

        featured_data = engineer.build_features()

        logger.info(
            "Feature engineering completed."
        )

        # ==========================================================
        # Step 5 : Forecast Future Sales
        # ==========================================================

        forecaster = ChronosForecaster(
            featured_data
        )

        predictions = forecaster.run()

        logger.info(
            "Sales forecasting completed."
        )

        # ==========================================================
        # Step 6 : Model Evaluation
        # ==========================================================

        evaluator = ForecastEvaluator(

            actual=featured_data[TARGET_COLUMN]
            [-len(predictions):],

            predicted=predictions

        )

        metrics = evaluator.evaluate()

        logger.info(metrics)

        print("\nForecast Evaluation\n")

        for metric, value in metrics.items():

            print(f"{metric:<12}: {value}")

        # ==========================================================
        # Step 7 : Visualization
        # ==========================================================

        visualizer = SalesVisualizer()

        visualizer.plot_sales_history(
            featured_data
        )

        visualizer.plot_forecast(
            predictions
        )

        visualizer.plot_actual_vs_predicted(

            featured_data[TARGET_COLUMN]
            [-len(predictions):],

            predictions

        )

        visualizer.plot_monthly_sales(
            featured_data
        )

        logger.info(
            "Visualization reports generated."
        )

    except Exception as error:

        logger.exception(

            f"Unexpected application error: {error}"

        )

    finally:

        database.disconnect()

        logger.info("=" * 80)

        logger.info(
            "Forecasting Pipeline Finished Successfully"
        )

        logger.info("=" * 80)

        print("\nApplication Finished Successfully.")


# =============================================================================
# Application Entry Point
# =============================================================================

if __name__ == "__main__":

    main()